In [22]:
import pandas as pd
import argos
from argos import ArgosDataset

## Importando o csv

In [3]:
metadata = pd.read_csv("../data/raw/isolates_metadata.csv")

print(metadata.shape)
metadata.head()

(8381, 8)


,Species,Source,Date,Location,sample,BioSample,Estado,Região
0,Leptospira interrogans,Homo sapiens,NaN,Salvador,GCA_000216055.2,SAMN00254327,BA,Nordeste
1,Escherichia coli,NaN,1990.0,Brazil,GCA_000316425.1,SAMN01041333,NaN,NaN
2,Vibrio cholerae,patient with cholera-like diarrhea,1991.0,NaN,GCA_000223095.2,SAMN02470783,NaN,NaN
3,Acinetobacter bereziniae,Rectal Swab,2019.0,Brazil,GCA_036761135.1,SAMN39408214,NaN,NaN
4,Acinetobacter bereziniae,Endotracheal aspirate,2014.0,"Londrina, PR",GCA_003670255.1,SAMN09907131,PR,Sul


## Verificando as espécies

In [4]:
unique_species = list(set(metadata['Species'].to_list()))

print(unique_species)
print(len(unique_species))

['Mycobacteroides saopaulense', 'Acinetobacter courvalinii', 'Rhodopirellula sp.', 'Proteus mirabilis', 'Streptococcus vestibularis', 'Aeromonas australiensis', 'Bordetella trematum', 'Corynebacterium aurimucosum', 'Pluralibacter gergoviae', 'Campylobacter coli', 'Mycobacteroides abscessus', 'Serratia marcescens', 'Lactobacillus crispatus', 'Staphylococcus coagulans', 'Enterobacter kobei', 'Oxynema sp.', 'Candidozyma auris', 'Enterococcus faecalis', 'Staphylococcus lugdunensis', 'Helicobacter pylori', 'Brucella oryzae', 'Raoultella ornithinolytica', 'Enterobacter chengduensis', 'Haemophilus influenzae', 'Synergistota bacterium', 'Bifidobacterium longum', 'Acinetobacter pittii', 'Elizabethkingia miricola', 'Acinetobacter colistiniresistens', 'Elizabethkingia meningoseptica', 'Neisseria sp.', 'Staphylococcus aureus', 'Haemophilus pittmaniae', 'Corynebacterium minutissimum', 'Mycobacterium tuberculosis', 'Enterobacter hormaechei', 'Corynebacterium diphtheriae', 'Aggregatibacter actinomyce

In [5]:
metadata.value_counts('Species')

Species
Neisseria gonorrhoeae         1310
Klebsiella pneumoniae         1179
Salmonella enterica            735
Escherichia coli               531
Bacillota bacterium            506
                              ... 
Nocardia otitidiscaviarum        1
Staphylococcus lugdunensis       1
Staphylococcus capitis           1
Staphylococcus sp.               1
Vibrio vulnificus                1
Name: count, Length: 161, dtype: int64

In [10]:
# ---------------------------------------------------------------------
# WHO Bacterial Priority Pathogens List (BPPL) 2024
# Fonte: WHO Bacterial Priority Pathogens List, 2024 update (IRIS/WHO,
# ISBN 978-92-4-009346-3) + The Lancet Infectious Diseases (2025),
# "The WHO Bacterial Priority Pathogens List 2024: a prioritisation study"
# ---------------------------------------------------------------------

# Exceções explícitas -- a própria OMS tira Salmonella/Shigella do bloco
# genérico de Enterobacterales e realoca pra High (não fica em Critical
# só porque é Enterobacterales)
_ESPECIE_EXATA = {
    "acinetobacter baumannii": "Critical",   # Moraxellaceae, NÃO Enterobacterales -- OMS trata à parte
    "mycobacterium tuberculosis": "Critical",

    "enterococcus faecium": "High",
    "staphylococcus aureus": "High",
    "helicobacter pylori": "High",
    "pseudomonas aeruginosa": "High",
    "neisseria gonorrhoeae": "High",

    "streptococcus pneumoniae": "Medium",
    "haemophilus influenzae": "Medium",
    "streptococcus pyogenes": "Medium",     # Grupo A
    "streptococcus agalactiae": "Medium",   # Grupo B
}

_GENERO_EXATO = {
    "campylobacter": "High",   # "Campylobacter spp" -- a OMS não restringe a 1 espécie
}

# Gêneros de Enterobacterales -- Critical por padrão, MENOS Salmonella/Shigella
_GENERO_ENTEROBACTERALES_CRITICAL = {
    "escherichia", "klebsiella", "enterobacter", "citrobacter", "kluyvera",
    "serratia", "leclercia", "raoultella", "providencia", "morganella",
    "proteus", "pluralibacter", "pseudocitrobacter", "yersinia", "cronobacter",
}
_GENERO_ENTEROBACTERALES_HIGH = {"salmonella", "shigella"}

# Não é BPPL 2024 (lista é só bactéria) -- C. auris está na lista de FUNGOS
# da OMS (WHO Fungal Priority Pathogens List, 2022), onde é Critical --
# tier diferente, não dá pra misturar na mesma coluna sem confundir
_FUNGO_CONHECIDO = {"candidozyma", "candida"}


def classificar_who_priority(Species):
    if pd.isna(Species):
        return "Unknown"
    s = str(Species).strip().lower()
    genero = s.split()[0] if s else ""

    if genero in _FUNGO_CONHECIDO:
        return "Not applicable (fungo -- ver WHO Fungal Priority Pathogens List)"

    if s in _ESPECIE_EXATA:
        return _ESPECIE_EXATA[s]

    if genero in _GENERO_EXATO:
        return _GENERO_EXATO[genero]

    if genero in _GENERO_ENTEROBACTERALES_HIGH:
        return "High"
    if genero in _GENERO_ENTEROBACTERALES_CRITICAL:
        return "Critical"

    return "Other"

In [11]:
metadata["WHO_Priority"] = metadata["Species"].apply(classificar_who_priority)
metadata["WHO_Priority"].value_counts()

WHO_Priority
High                                                                3203
Critical                                                            2463
Other                                                               2091
Medium                                                               614
Not applicable (fungo -- ver WHO Fungal Priority Pathogens List)      10
Name: count, dtype: int64

## Verificando as Sources #terror

In [6]:
unique_sources = list(set(metadata['Source'].to_list()))

print(unique_sources)
len(unique_sources) 

['Perianal Swab', 'Biological material', 'Nasal secretion', 'Scarlet fever', 'respiratory tract infection', 'patient with blood stream infection', 'sputum from cystic fibrosis patients', 'Punch Liquid', 'milk sample', 'Tracheal Secretion', 'patient subjected to laparoscopic cholecystectomy', 'swab of secretion from the foot', 'Venous cateter', 'Bacteremia', 'Liver Fluid', 'Inferior respiratory tract', 'Blood Culture', 'peritoneal secretion', 'pleural liquid', 'clinical (blood newborn sepsis)', 'urinary tract', 'CSF', 'hand', 'wound', 'sputum from cystic fibrosis', 'Human tissue', 'clinical (CSF, meningitis)', 'Anal/vaginal swab', 'adult', 'BAL', 'abscess', 'PLEURAL FLUID', 'Blood stream', 'Pleral fluid', 'GU: Urine', 'catheter tip', 'swab', 'Nostril', 'From patient with pneumonia', 'nares', 'Fecal Swab', 'oral cavity, infant (15 months age)', 'Abscess', 'tissue', 'Blood', 'catheter bridge', 'clinical (placenta)', 'food', 'sacral ulcer', 'Knee discharge', 'biopsy of vertebra', 'bone mar

379

In [8]:
def classificar_source_type(source):
    if pd.isna(source):
        return "Unknown"
    s = str(source).lower().strip()

    if s in {"not available", "na", "n/a", "not known", "nd", "unknown", "-",
              "not collected", "other/unknown", "others", "other", "not been collected"}:
        return "Unknown"

    ambiental_kw = ["river", "mangrove", "soil", "sediment", "sewer", "effluent"]
    alimento_animal_kw = ["food", "chicken", "poultry", "milk sample"]
    if any(k in s for k in ambiental_kw):
        return "Environmental"
    if any(k in s for k in alimento_animal_kw):
        return "Food/Animal"

    if any(k in s for k in ["blood", "bloodstream", "hemocultur", "catether (blood)", "bacteremia"]):
        return "Blood"
    if any(k in s for k in ["urin", "gu:"]):
        return "Urinary"
    if any(k in s for k in ["respirat", "bronch", "trache", "sputum", "lung", "pleural",
                              "bal", "pneumonia", "throat", "pharyn", "nasal", "nose",
                              "nostril", "nares", "oropharynx"]):
        return "Respiratory"
    if any(k in s for k in ["stool", "fecal", "feces", "faeces", "rectal", "gastr",
                              "gi:", "abdominal", "peritoneal", "ascit", "bile",
                              "gallbladder", "duodenum", "diarrhea"]):
        return "Gastrointestinal"
    if any(k in s for k in ["csf", "cerebrospinal", "liquor", "brain", "meningitis"]):
        return "CNS"
    if any(k in s for k in ["catheter", "catether", "prosthesis", "device", "suture", "sonication"]):
        return "Medical Device"
    if any(k in s for k in ["wound", "ulcer", "abscess", "skin", "soft tissue", "secretion",
                              "tissue", "biopsy", "knee", "hip", "femur"]):
        return "Skin & Soft Tissue"
    if any(k in s for k in ["vagin", "cervi", "genital", "semen", "urethra", "oral cavity",
                              "dental", "ear", "eye", "conjunctiv"]):
        return "Body Fluid"

    return "Other"

metadata["source_type"] = metadata["Source"].apply(classificar_source_type)
metadata["source_type"].value_counts()

source_type
Gastrointestinal      1919
Blood                 1777
Unknown               1242
Other                  988
Urinary                666
Respiratory            663
CNS                    530
Food/Animal            178
Body Fluid             168
Skin & Soft Tissue     163
Medical Device          77
Environmental           10
Name: count, dtype: int64

## Filtragem

In [ ]:
metadata['source_type'].unique()

array(['Other', 'Unknown', 'Gastrointestinal', 'Respiratory', 'Blood',
       'Body Fluid', 'Skin & Soft Tissue', 'Environmental', 'Food/Animal',
       'Medical Device', 'Urinary', 'CNS'], dtype=object)

In [18]:
metadata['WHO_Priority'].unique()

array(['Other', 'Critical', 'High', 'Medium',
       'Not applicable (fungo -- ver WHO Fungal Priority Pathogens List)'],
      dtype=object)

In [19]:
keep_sources = [
    'Other', 'Unknown', 'Gastrointestinal', 'Respiratory', 'Blood',
    'Body Fluid', 'Skin & Soft Tissue', 'Medical Device', 'Urinary', 'CNS'
    ]

keep_who = [
    'Other', 'Critical', 'High', 'Medium'
    ]

# Erro no geNomad
ids_sem_contigs = [
    "GCA_000229105.2", "GCA_000230165.2", "GCA_000230185.2", "GCA_000230225.2",
    "GCA_015959325.1", "GCA_023454935.1", "GCA_023454965.1", "GCA_029076025.1",
    "GCA_042286705.1", "GCA_042286735.1",
]

In [25]:
dataset = ArgosDataset.from_parquet("../data/argos_project/parquet_data")
ids_no_projeto = set(dataset.metadata.index)

n_bruto = len(metadata)
metadata_projeto = metadata[metadata["sample"].isin(ids_no_projeto)].copy()
print(f"{n_bruto} linhas no CSV bruto -> {len(metadata_projeto)} que de fato estão no argos_project")
print(f"({n_bruto - len(metadata_projeto)} nunca chegaram a virar genoma processado por baixa completude/N50)")

8381 linhas no CSV bruto -> 7065 que de fato estão no argos_project
(1316 nunca chegaram a virar genoma processado por baixa completude/N50)


In [26]:
mask_source_ruim = ~metadata_projeto["source_type"].isin(keep_sources)
mask_fungo = ~metadata_projeto["WHO_Priority"].isin(keep_who)
mask_sem_contigs = metadata_projeto["sample"].isin(ids_sem_contigs)

print(f"Source ambiental/alimento: {mask_source_ruim.sum()}")
print(f"Fungo: {mask_fungo.sum()}")
print(f"Sem contigs: {mask_sem_contigs.sum()}")

metadata_filtrado = metadata_projeto[~(mask_source_ruim | mask_fungo | mask_sem_contigs)].copy()
print(f"\n{len(metadata_projeto)} -> {len(metadata_filtrado)} genomas ({len(metadata_projeto) - len(metadata_filtrado)} removidos)")

Source ambiental/alimento: 15
Fungo: 2
Sem contigs: 10

7065 -> 7038 genomas (27 removidos)


In [27]:
metadata_filtrado.head()

,Species,Source,Date,Location,sample,BioSample,Estado,Região,source_type,WHO_Priority
0,Leptospira interrogans,Homo sapiens,NaN,Salvador,GCA_000216055.2,SAMN00254327,BA,Nordeste,Other,Other
1,Escherichia coli,NaN,1990.0,Brazil,GCA_000316425.1,SAMN01041333,NaN,NaN,Unknown,Critical
2,Vibrio cholerae,patient with cholera-like diarrhea,1991.0,NaN,GCA_000223095.2,SAMN02470783,NaN,NaN,Gastrointestinal,Other
3,Acinetobacter bereziniae,Rectal Swab,2019.0,Brazil,GCA_036761135.1,SAMN39408214,NaN,NaN,Gastrointestinal,Other
4,Acinetobacter bereziniae,Endotracheal aspirate,2014.0,"Londrina, PR",GCA_003670255.1,SAMN09907131,PR,Sul,Respiratory,Other


In [29]:
ids_finais = metadata_filtrado["sample"].tolist()
dataset = ArgosDataset.from_parquet("../data/argos_project/parquet_data", samples=ids_finais)
dataset.metadata["Species"].value_counts()

Species
Klebsiella pneumoniae           1056
Salmonella enterica              558
Escherichia coli                 529
Bacillota bacterium              504
Staphylococcus aureus            475
                                ... 
Enterococcus hirae                 1
Corynebacterium minutissimum       1
Haemophilus pittmaniae             1
Mycobacterium kyorinense           1
Staphylococcus sp.                 1
Name: count, Length: 152, dtype: int64

## Salvando os metadados filtrados

In [31]:
metadata_filtrado.to_csv("../data/processed/metadata_filtrada.csv", index=False)